# 01. ??????????? ?????? ?????? ? strict contract

**???? ???????.** ????????, ?? ???? ??????? ???????? ????? `raw_data`, ??? ?? ???????????? ? ??????? ??????? ???????? ? ?????? ?????????? ?????? ?????? ????????? workload, ????????? ?????????????? ? ?????????? `????? / ?????????????? / ????`.

???? notebook ?? ??????? ????????? ??????. ?? ???????? ?? ??????: ?????????? ?? ????????? ? ?????????????? ???????? ?????????? ?????? ??? ??????????? baseline ? Efficient HADES.


In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path.cwd()
ARTIFACT_DIR = PROJECT_ROOT / "dashboard" / "artifacts"
CONTRACT_PATH = ARTIFACT_DIR / "contract" / "strict_intensity_contract.csv.gz"
REPORT_DIR = ARTIFACT_DIR / "reports"
BASELINE_DIR = REPORT_DIR / "protocol_fixed_baselines"
MODEL_OUT_DIR = ARTIFACT_DIR / "model" / "protocol_fixed_efficient_hades"
RUN_ROOT = ARTIFACT_DIR / "model_runs" / "protocol_fixed_hades"
RUN_ID = "final_protocol_fixed_hades"
BUNDLE_DIR = ARTIFACT_DIR / "bundle"

def run_command(command: list[str]) -> None:
    print(" ".join(command))
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)

def read_json(path: Path) -> dict:
    if not path.exists():
        return {}
    return json.loads(path.read_text(encoding="utf-8"))

def display_existing(path: Path, reader=pd.read_csv, **kwargs):
    if not path.exists():
        print(f"???? ???? ?? ??????: {path.relative_to(PROJECT_ROOT)}")
        return pd.DataFrame()
    frame = reader(path, **kwargs)
    display(frame)
    return frame


## 1. ???????? ??????

**???? ???????.** ?????????, ??? ????????? ?????? ?????????? ?????? ? `raw_data`, ? ?? ?? ?????? ????????????? outputs. ??? ?????????? workload, ???? MapReduce ? KPI-????? ?? ?????????.


In [ ]:
RAW_DATA = PROJECT_ROOT / "raw_data"
workloads = sorted([p for p in RAW_DATA.iterdir() if p.is_dir()]) if RAW_DATA.exists() else []
rows = []
for workload in workloads:
    log_files = list((workload / "mapred_logs").glob("*.log"))
    kpi_files = list((workload / "kpis").rglob("*"))
    kpi_files = [p for p in kpi_files if p.is_file()]
    scenarios = sorted({p.parts[len((workload / "kpis").parts)] for p in kpi_files if len(p.parts) > len((workload / "kpis").parts)}) if (workload / "kpis").exists() else []
    rows.append({
        "workload": workload.name,
        "log_files": len(log_files),
        "kpi_files": len(kpi_files),
        "kpi_scenarios": len(scenarios),
    })
raw_inventory = pd.DataFrame(rows).sort_values("workload")
display(raw_inventory)
ax = raw_inventory.set_index("workload")[["log_files", "kpi_files"]].plot(kind="bar", figsize=(10, 4), title="?????? raw_data ?? workload")
ax.set_xlabel("Workload")
ax.set_ylabel("?????????? ??????")
plt.tight_layout()


**????????? ?????.** ??????? ???? ????????? ?????????? ???????? ??????. ???? workload ???????????? ????????????, ??????? random/current split ????? ????????? domain-?????? ? anomaly-????????; ??? ? ????? ????? ?? ?????? ???????? ? protocol-fixed ??????.


## 2. ?????????? strict contract

**???? ???????.** ???????? ?????? ??????? ??????? ? ??????, KPI, split ? ???????? ???????. ???????? ???????? python-???????, ? notebook ?????? ????????? ? ????????? ?????????.


In [ ]:
BUILD_CONTRACT_IF_MISSING = True
if BUILD_CONTRACT_IF_MISSING and not CONTRACT_PATH.exists():
    run_command([sys.executable, "-m", "dashboard.pipeline.build_contract", "--raw-data", "raw_data"])

contract_summary = display_existing(ARTIFACT_DIR / "contract" / "strict_contract_summary.csv")
contract_report = read_json(REPORT_DIR / "final_data_pipeline_report.json")
print(json.dumps({k: contract_report.get(k) for k in ["contract_path", "rows", "run_count", "workload_count"]}, ensure_ascii=False, indent=2))


In [ ]:
contract = pd.read_csv(CONTRACT_PATH, low_memory=False)
contract["window_ts"] = pd.to_datetime(contract["window_ts"], utc=True, errors="coerce")
if "window_end" in contract.columns:
    contract["window_end"] = pd.to_datetime(contract["window_end"], utc=True, errors="coerce")
if "y_anomaly" not in contract.columns and {"y_warning", "y_risk"}.issubset(contract.columns):
    contract["y_anomaly"] = np.maximum(contract["y_warning"].astype(int), contract["y_risk"].astype(int))
contract.head(8)


**????????? ?????.** Contract stage ?????? ??????? ??????? ????: ???????? ??????? ?? ???? ??????????, ? ????? ?????????????? ? ????? ????????? ? ?????? gap ????? ????????. ?????? baseline ? Efficient HADES ?????????? ???? ? ??? ?? contract.


## 3. ???????? ? ?????????

**???? ???????.** ????????, ??? `y_warning`, `y_risk` ? horizon-???? ?????? ?????? ??????. ??????? ????????? ??????? ?? ?????? ?????? ????? ?????? ?? `y_anomaly`.


In [ ]:
target_cols = [c for c in ["y_warning", "y_risk", "y_anomaly", "horizon_30s", "horizon_60s", "horizon_120s", "horizon_300s"] if c in contract.columns]
if target_cols:
    target_summary = contract.groupby("split")[target_cols].mean().T
    display((target_summary * 100).round(2).rename_axis("target").reset_index())
    ax = (target_summary * 100).plot(kind="bar", figsize=(11, 4), title="???? positive-???? ?? split, %")
    ax.set_xlabel("Target")
    ax.set_ylabel("Positive windows, %")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
else:
    print("Horizon columns ??????????? ? contract; ??? ????? ????????? ? protocol-fixed training module.")


**????????? ?????.** ????? ??????? ????????? ?????? ????? ?????? positive-???? ? ????? ????????. Short-horizon risk (`30s/60s`) ??????? ??? ??????? ??????, ?? ?? ?????? ? ??????? ?????????? ??????? precision/recall.


## 4. Workload/domain shift

**???? ???????.** ?????????, ?? ???? ?? ?????? workload ?????? anomaly. ??? ????? ??????? ????????????? ????, positive-rate ? split ?? workload.


In [ ]:
workload_cols = ["workload", "split"] + [c for c in ["y_warning", "y_risk", "y_anomaly"] if c in contract.columns]
workload_summary = contract[workload_cols].groupby(["workload", "split"]).agg(
    rows=("split", "size"),
    warning_rate=("y_warning", "mean") if "y_warning" in contract.columns else ("split", "size"),
    risk_rate=("y_risk", "mean") if "y_risk" in contract.columns else ("split", "size"),
).reset_index()
display(workload_summary)

pivot_rows = workload_summary.pivot(index="workload", columns="split", values="rows").fillna(0)
ax = pivot_rows.plot(kind="bar", stacked=True, figsize=(10, 4), title="???? ?? workload ? split")
ax.set_xlabel("Workload")
ax.set_ylabel("?????????? ????")
plt.tight_layout()


**????????? ?????.** ?????? ????? ??????????? ???? domain shift: ???? split ?????????? ????????? workload ?? test ??? validation, ?????? ????? ??????????? ?? ??-?? ???????????, ? ??-?? ????????? ??????. ??????? ?????? ?????????? `stratified_by_workload`.


## 5. ??????????? ????????????

**???? ???????.** ?????????, ????????? ????? ? ????? ???? ???????????? ???????? ???? ? KPI. ??? ????? ??? ??????? ??????????????? fusion-??????.


In [ ]:
availability_cols = [c for c in ["log_available", "kpi_available", "kpi_series_available", "has_both_modalities"] if c in contract.columns]
if availability_cols:
    availability = contract.groupby("split")[availability_cols].mean().T * 100
    display(availability.round(2).rename_axis("feature").reset_index())
    ax = availability.plot(kind="bar", figsize=(10, 4), title="??????????? ????? ? KPI ?? split, %")
    ax.set_xlabel("??????? ???????????")
    ax.set_ylabel("????, %")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
else:
    print("? contract ??? ????? ??????? ??????????? ????????????.")


**????????? ?????.** Fusion-?????? ?????? ????? ???????? ? ????????? ??????. Dashboard ???????? ?????????? score ??????, alert-layer ? ???? incident, ????? ?? ????????? ???????????? ? ????????.


## ???? 01

Contract ????????? ??????????????? ??????????: `raw_data -> strict_intensity_contract`. ??????? ????? EDA: ???????? ????? ????????? ?? ?????? window-level F1, ?? ? ?? workload, ????????? ? alert-level. ??????? ????????? notebook ?????? baseline ??? ? protocol-fixed ??????????.
